In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

In [3]:
NUM_USERS = 20000
START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 12, 31)
np.random.seed(42)
random.seed(42)

# Creating output folder
OUTPUT_DIR = "ecom_sql_project"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
# Users Table
countries = ["India", "USA", "UK", "Canada", "Australia"]
country_prob = [0.65, 0.15, 0.10, 0.06, 0.04]

user_ids = [f"U{100000+i}" for i in range(NUM_USERS)]
signup_days = np.random.randint(0, (END_DATE - START_DATE).days, NUM_USERS)
signup_dates = [START_DATE + timedelta(days=int(x)) for x in signup_days]
user_country = np.random.choice(countries, size=NUM_USERS, p=country_prob)

users = pd.DataFrame({
    "user_id": user_ids,
    "signup_date": [d.date() for d in signup_dates],
    "country": user_country
})


In [5]:
# Products Table
categories = ["Electronics", "Apparel", "Home", "Beauty", "Grocery"]
product_count = 80

products = pd.DataFrame({
    "product_id": [f"P{500+i}" for i in range(product_count)],
    "category": np.random.choice(categories, size=product_count),
    "price": np.round(np.random.uniform(100, 8000, product_count), 2)
})

In [6]:
# Sessions Table
session_list = []
device_prob = [0.72, 0.23, 0.05]
utm_sources = ["organic", "email", "paid_social", "affiliate", "ads"]
utm_prob = [0.55, 0.15, 0.15, 0.10, 0.05]

session_id = 1
for uid, sdate in zip(user_ids, signup_dates):
    num_sessions = np.random.poisson(lam=3)

    for _ in range(num_sessions):
        sess_date = sdate + timedelta(days=random.randint(0, 364))
        if sess_date > END_DATE:
            continue

        session_list.append({
            "session_id": f"S{session_id}",
            "user_id": uid,
            "session_date": sess_date.date(),
            "device": np.random.choice(["Mobile", "Desktop", "Tablet"], p=device_prob),
            "utm_source": np.random.choice(utm_sources, p=utm_prob)
        })
        session_id += 1

sessions = pd.DataFrame(session_list)

In [7]:
# Add-to-cart Table
add_cart_list = []
cart_id = 1

for _, row in sessions.iterrows():
    if np.random.rand() < 0.35:  # 35% add-to-cart probability
        prod = products.sample(1).iloc[0]
        add_cart_list.append({
            "cart_id": f"C{cart_id}",
            "session_id": row["session_id"],
            "user_id": row["user_id"],
            "product_id": prod["product_id"],
            "cart_date": row["session_date"]
        })
        cart_id += 1

add_to_cart = pd.DataFrame(add_cart_list)

In [8]:
# Stock-out Table
stock_list = []
for _, prod in products.iterrows():
    if np.random.rand() < 0.25:  # 25% products had stockouts
        stock_date = START_DATE + timedelta(days=random.randint(0, 364))
        stock_list.append({
            "product_id": prod["product_id"],
            "stockout_date": stock_date.date(),
            "stockout_reason": np.random.choice(["High demand", "Supplier delay", "Inventory error"])
        })

stock_out = pd.DataFrame(stock_list)

In [9]:
# Orders Table
orders_list = []
order_id = 1

for _, row in add_to_cart.iterrows():
    # Only some cart items convert to orders
    if np.random.rand() < 0.40:  # 40% cart -> purchase
        prod = products[products["product_id"] == row["product_id"]].iloc[0]

        # Check if product was in stock
        stock_issue = False
        if row["product_id"] in stock_out["product_id"].values:
            stock_issue = True if np.random.rand() < 0.30 else False

        if stock_issue:
            continue  # No purchase due to stockout

        qty = np.random.choice([1, 1, 2, 3], p=[0.7, 0.2, 0.08, 0.02])
        orders_list.append({
            "order_id": f"O{order_id}",
            "user_id": row["user_id"],
            "order_date": row["cart_date"],
            "product_id": row["product_id"],
            "quantity": qty,
            "price": prod["price"],
            "order_value": round(prod["price"] * qty, 2)
        })
        order_id += 1

orders = pd.DataFrame(orders_list)

In [10]:
# Marketing Table 
date_range = pd.date_range(START_DATE, END_DATE, freq="D")

n = len(date_range)  

marketing = pd.DataFrame({
    "date": date_range,
    "utm_source": np.random.choice(utm_sources, size=n, p=utm_prob),
    "ad_spend": np.random.uniform(2000, 80000, size=n).round(2)
})

In [11]:
# Saving Tables
users.to_csv(f"{OUTPUT_DIR}/users.csv", index=False)
sessions.to_csv(f"{OUTPUT_DIR}/sessions.csv", index=False)
products.to_csv(f"{OUTPUT_DIR}/products.csv", index=False)
add_to_cart.to_csv(f"{OUTPUT_DIR}/add_to_cart.csv", index=False)
orders.to_csv(f"{OUTPUT_DIR}/orders.csv", index=False)
stock_out.to_csv(f"{OUTPUT_DIR}/stock_out.csv", index=False)
marketing.to_csv(f"{OUTPUT_DIR}/marketing.csv", index=False)

In [12]:
print("Dataset generated successfully!")
print("Files saved in folder:", OUTPUT_DIR)
print("Rows:")
print("Users:", users.shape)
print("Sessions:", sessions.shape)
print("Add-to-Cart:", add_to_cart.shape)
print("Orders:", orders.shape)
print("Stock Out:", stock_out.shape)
print("Marketing:", marketing.shape)
print("Products:", products.shape)

Dataset generated successfully!
Files saved in folder: ecom_sql_project
Rows:
Users: (20000, 3)
Sessions: (30395, 5)
Add-to-Cart: (10674, 5)
Orders: (3826, 7)
Stock Out: (27, 3)
Marketing: (366, 3)
Products: (80, 3)


In [25]:
pip install pymysql

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [27]:
from sqlalchemy import create_engine
import mysql.connector

# Database Connection Details
DB_NAME = 'ecommerce_project'
DB_USER = 'shreya'
DB_PASSWORD = 'Shreya620'
DB_HOST = 'localhost'
DB_PORT = 3306

MYSQL_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Create Engine
try:
    engine = create_engine(MYSQL_URL)
    print("Database connection engine created successfully.\n")
except Exception as e:
    print(f" Error creating engine: {e}")
    exit()

# Data Loading Function
CSV_PATH = r"C:/Users/Shreya Ranjan/E-comm Project/ecom_sql_project" 

# Data Loading Function

def load_data_to_mysql(filename, table_name, index_status=False):
    filepath = f"{CSV_PATH}/{filename}"
    print(f" Reading {filepath}...")

    try:
        df = pd.read_csv(filepath)
        df.to_sql(
            name=table_name,
            con=engine,
            if_exists='append',
            index=index_status
        )
        print(f" Loaded {len(df)} rows into '{table_name}'.\n")
    except Exception as e:
        print(f" Error loading data into '{table_name}': {e}\n")


load_data_to_mysql('users.csv', 'users')
load_data_to_mysql('sessions.csv', 'sessions')
load_data_to_mysql('products.csv', 'products')
load_data_to_mysql('add_to_cart.csv', 'add_to_cart')
load_data_to_mysql('orders.csv', 'orders')
load_data_to_mysql('stock_out.csv', 'stock_out')
load_data_to_mysql('marketing.csv', 'marketing')

print("\nAll data loading completed successfully!")

Database connection engine created successfully.

 Reading C:/Users/Shreya Ranjan/E-comm Project/ecom_sql_project/users.csv...
 Loaded 20000 rows into 'users'.

 Reading C:/Users/Shreya Ranjan/E-comm Project/ecom_sql_project/sessions.csv...
 Loaded 30395 rows into 'sessions'.

 Reading C:/Users/Shreya Ranjan/E-comm Project/ecom_sql_project/products.csv...
 Loaded 80 rows into 'products'.

 Reading C:/Users/Shreya Ranjan/E-comm Project/ecom_sql_project/add_to_cart.csv...
 Loaded 10674 rows into 'add_to_cart'.

 Reading C:/Users/Shreya Ranjan/E-comm Project/ecom_sql_project/orders.csv...
 Loaded 3826 rows into 'orders'.

 Reading C:/Users/Shreya Ranjan/E-comm Project/ecom_sql_project/stock_out.csv...
 Loaded 27 rows into 'stock_out'.

 Reading C:/Users/Shreya Ranjan/E-comm Project/ecom_sql_project/marketing.csv...
 Loaded 366 rows into 'marketing'.


All data loading completed successfully!
